<a href="https://colab.research.google.com/github/VincenzoDamico/ProteinDynamics/blob/LSTM/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



# Library setting



In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from sklearn.preprocessing import StandardScaler
import os
import random

Seeting the seeds:

In [3]:
# fix random seed for reproducibility
def set_all_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_all_seeds(42)
np.random.seed(42) #it also fix the seed for sklearn

# Inizialization of Input Data

In [4]:
driver_path='/content/drive/MyDrive/ProteinDynamics/'

In [5]:
coordinates_df = pd.read_csv(driver_path+'coordinates.csv', header=None, index_col=False, sep=';')
# header= none: we don't have col names in the first raw
# index_col: we don't have a col for the index
print(coordinates_df.head(), coordinates_df.shape)

# Forces:
forces_df_raw = pd.read_csv(driver_path+'forces.csv', header=None, index_col=False, sep=None, engine='python')
# sep = none means pandas will search to find the separator
forces_df = forces_df_raw[0].str.split('\t', expand=True)
forces_df = forces_df.apply(pd.to_numeric, errors='coerce')
print(forces_df.head(), forces_df.shape)

# velocity:
velocities_df_raw = pd.read_csv(driver_path+'velocity.csv', header=None, index_col=False, sep=None, engine='python')
velocities_df = velocities_df_raw[0].str.split('\t', expand=True)
velocities_df = velocities_df.apply(pd.to_numeric, errors='coerce')
print(velocities_df.head(), velocities_df.shape)

     0        1        2        3        4        5        6        7    \
0  0.000  56.3980  51.9370  52.6880  56.3750  52.0050  52.6170  56.4260   
1  0.002  56.3992  51.9368  52.6877  56.3743  52.0037  52.6162  56.4267   
2  0.004  56.3999  51.9366  52.6871  56.3767  52.0046  52.6161  56.4285   
3  0.006  56.4001  51.9364  52.6861  56.3816  52.0073  52.6167  56.4315   
4  0.008  56.4001  51.9362  52.6848  56.3876  52.0109  52.6179  56.4350   

       8        9    ...      405      406      407      408      409  \
0  51.8530  52.6400  ...  52.1880  56.7460  52.0040  52.3850  56.7650   
1  51.8556  52.6344  ...  52.1872  56.7448  52.0025  52.3849  56.7651   
2  51.8579  52.6307  ...  52.1890  56.7434  52.0011  52.3842  56.7652   
3  51.8594  52.6288  ...  52.1930  56.7417  52.0001  52.3832  56.7652   
4  51.8597  52.6288  ...  52.1982  56.7402  51.9995  52.3824  56.7651   

       410      411      412      413      414  
0  51.9040  52.4540  56.7550  52.1200  52.4210  
1  51.9044  


LSTMs are sensitive to the scale of the input data, specifically when the sigmoid (default) or tanh activation functions are used. It can be a good practice to rescale the data to the range of 0-to-1



In [6]:
NUM_ATOMS = 138
feaature_xyz=3
num_features = NUM_ATOMS * feaature_xyz*3

In [7]:
coords = coordinates_df.iloc[:, 1:].values
vels   = velocities_df.iloc[:, 1:].values
forces = forces_df.iloc[:, 1:].values

coords = coords.reshape(-1,NUM_ATOMS, feaature_xyz)
vels   = vels.reshape(-1, NUM_ATOMS, feaature_xyz)
forces = forces.reshape(-1, NUM_ATOMS, feaature_xyz)

#3- Concatenate Features to be [x,y,z,vx,vy,vz,fx,fy,fz]
features = np.concatenate(
    [coords, vels, forces],
    axis=-1
)

In [8]:
N= len(features)                # 50001
train_end = int(N * 0.70)   # 35000
val_end = int(N * 0.85)         # 42500

In [9]:
features_flattened = features.reshape(N, num_features)
scaler = StandardScaler()
scaler.fit(features_flattened[:train_end])
features_scaled = scaler.transform(features_flattened)

In [10]:
print(features_scaled.shape)

(50001, 1242)


Adjust the input data to match the LMTS Model standard. The LSTM network expects the input data (X) to be provided with a specific array structure in the form of [samples, time steps, features].


# Prediction

In [16]:
look_back=5
epochs= 40
batchs_size=64

In [17]:
nb_samples = len(features_scaled)

pad_rows = np.zeros((look_back - 1, num_features))
features_padded = np.vstack((pad_rows, features_scaled))

X_reshaped = np.zeros((nb_samples, look_back, num_features))
Y_reshaped = np.zeros((nb_samples ,num_features))

for i in np.arange(0,nb_samples):
    in_pos = i + look_back
    X_reshaped[i] = features_padded[i:i+look_back].reshape(look_back, num_features)
    Y_reshaped[i] = features_scaled[i]



In [ ]:

trainX=X_reshaped[:train_end]
testX=X_reshaped[train_end:val_end]
valX=X_reshaped[val_end:]

trainY=Y_reshaped[:train_end]
testY=Y_reshaped[train_end:val_end]
valY=Y_reshaped[val_end:]

model = tf.keras.Sequential([
    # Bidirectional LSTM to capture patterns in both directions of the sequence
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False, dropout=0.2)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    # Final layer matches the 1242 features we want to predict
    # Default activation is 'linear', which is perfectly correct for regression
    tf.keras.layers.Dense(num_features)
])

# Compile using Mean Squared Error
model.compile(
    loss=tf.keras.losses.MeanSquaredError(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=[tf.keras.metrics.MeanAbsoluteError()] # Use MAE to track average error
)

# Train the model
history = model.fit(
    x=trainX, y=trainY,
    epochs=epochs,
    batch_size=batchs_size, # Batch size helps memory management and gradient calculation
    validation_data=(valX, valY)
)
trainPredict = model.predict(trainX)
# make predictions


Epoch 1/40
547/547 ━━━━━━━━━━━━━━━━━━━━ 36s 53ms/step - loss: 0.7354 - mean_absolute_error: 0.6165 - val_loss: 2.3244 - val_mean_absolute_error: 1.2536
Epoch 2/40
547/547 ━━━━━━━━━━━━━━━━━━━━ 41s 52ms/step - loss: 0.6746 - mean_absolute_error: 0.5629 - val_loss: 2.1375 - val_mean_absolute_error: 1.2070
Epoch 3/40
547/547 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - loss: 0.6677 - mean_absolute_error: 0.5564 - val_loss: 2.0251 - val_mean_absolute_error: 1.1774
Epoch 4/40
244/547 ━━━━━━━━━━━━━━━━━━━━ 15s 50ms/step - loss: 0.6633 - mean_absolute_error: 0.5535

In [ ]:
test_loss, test_acc = model.evaluate(testX,testY)

print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

In [ ]:

def plot_graphs(history, metric, title):
    # Pass labels directly to the plot function for a cleaner legend
    plt.plot(history.history[metric], label=f'Train {metric}')
    plt.plot(history.history['val_' + metric], label=f'Validation {metric}')
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6) # Adding a grid helps read continuous metrics

plt.figure(figsize=(16, 6))

# Subplot 1: Mean Absolute Error (Replaces Accuracy)
plt.subplot(1, 2, 1)
# Keras usually tracks MAE as 'mean_absolute_error' or 'mae' depending on the exact string used
metric_name = 'mean_absolute_error' if 'mean_absolute_error' in history.history else 'mae'
plot_graphs(history, metric_name, 'Model Error (MAE)')

# Subplot 2: Loss (Mean Squared Error)
plt.subplot(1, 2, 2)
plot_graphs(history, 'loss', 'Model Loss (MSE)')

# Optional: If your initial error is incredibly high and drops fast,
# the graph will look like an "L" shape. Uncommenting the line below
# puts the Y-axis on a log scale so you can see the fine details of the training curve.
# plt.yscale('log')

plt.tight_layout()
plt.savefig('training_curves.png') # Good practice to save the plot!
plt.show()